# 00 · Start here

Welcome. These notebooks are a **tutorial** on the forecasting experiment in this project: how a synthetic agricultural
market is built, how weekly wholesale prices are forecast 1, 2 and 4 weeks ahead, and - just as important - how
such an experiment can quietly go wrong.

**How the tutorials relate to the real project**

| | The pipeline (`src/`, `scripts/`) | These tutorials (`tutorials/`) |
|---|---|---|
| Purpose | Produce the experiment's results | Explain *why* each step exists |
| Code | The single source of truth | A simplified, standalone re-implementation (`tutorial_utils.py`) |
| Data | Generates it | **Reads** the shipped data and results - never writes to them |
| If they disagree | It is right | The tutorial gets fixed |

Numbers you compute here may differ slightly from the final report, because the tutorials use fewer features, smaller
model settings and shortcuts. Where we can, we print the pipeline's number next to ours so you can see how close they are.
**Only the pipeline's numbers are the experiment's results.**

> Everything is **synthetic**. Place names and KES prices are for readability only; nothing here describes real markets.

## Reading order

| # | Notebook | The question it answers |
|---|---|---|
| 01 | The forecasting problem | What exactly are we predicting, and how do we score it? |
| 02 | Building the synthetic world | Where do the prices come from, and what is "known truth"? |
| 03 | Data quality | What does messy data look like, and how do we clean it *without* cheating? |
| 04 | EDA with discipline | What can exploration tell us - and what must it not influence? |
| 05 | Leakage | How can a model look brilliant for the wrong reason? (deliberate leaks!) |
| 06 | Validation splits | Why chronological splits, purging and rolling origins? |
| 07 | Baselines | Why is "tomorrow equals today" so hard to beat? |
| 08 | Features and models | What do the ML models actually see, and what does each feature group add? |
| 09 | Tuning and statistics | How do we tune honestly and tell real differences from luck? |
| 10 | Stress tests and errors | When and why do the models fail? What would change on real data? |

Each notebook ends with **exercises**. Run them - the best way to understand a pipeline is to break it on purpose.

Total runtime for all notebooks is a few minutes on a laptop.

## Check your setup

In [1]:
import sys, platform
import numpy as np, pandas as pd, sklearn, matplotlib
from tutorial_utils import *
style()
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
print("project folder found:", ROOT.name)
for folder in ["data", "results", "config"]:
    n = len(list((ROOT / folder).glob("*")))
    print(f"  {folder:8s} {n} files")

python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1
project folder found: agri_price_experiment
  data     10 files
  results  45 files
  config   1 files


## A first look at the data
The main file is a **panel**: one row per market × crop × week (5 × 4 × 156 = 3,120 rows). We load the *realistic* scenario.

In [2]:
df = load_panel("realistic")
print(df.shape)
df[df.series == "MKT-01|CRP-02"].head(6).T

(3120, 23)


,156,157,158,159,160,161
week_start,2023-01-02,2023-01-09,2023-01-16,2023-01-23,2023-01-30,2023-02-06
week_index,1,2,3,4,5,6
market_id,MKT-01,MKT-01,MKT-01,MKT-01,MKT-01,MKT-01
crop_id,CRP-02,CRP-02,CRP-02,CRP-02,CRP-02,CRP-02
wholesale_price_kes_kg,92.05,95.14,93.91,91.9,90.9,80.86
latent_price_kes_kg,92.517,99.677,92.629,95.36,91.52,83.339
market_arrivals_tonnes,383.035,340.211,317.364,303.633,391.962,482.734
estimated_demand_index,132.593,130.416,126.762,125.458,124.539,132.741
rainfall_mm,19.139,12.376,22.957,7.624,11.274,6.66
temperature_c,19.896,20.06,20.8,20.118,20.486,19.744


Every column is explained in `docs/data_dictionary.md`. The ones to keep in mind from the start:

* `wholesale_price_kes_kg` - the **observed** price (this is what we forecast). It can be missing or wrong.
* `latent_price_kes_kg` - the **true** generator price. It exists only because the data are synthetic. It is an *audit* column:
  a model that uses it is cheating (notebook 05).
* `data_quality_flag` - `clean`, `missing`, `suspect` (an injected reporting error) or `synthetic_shock` (a real, documented event).
  Also an audit column: it tells *us* what happened to the data; a forecaster in real life would not have it.
* `report_delay_weeks` - 1 when the indicator values of that week reach the forecaster one week late.

## Glossary

* **Forecast origin (t)** - the end of week *t*: the moment a forecast is issued. Everything the model uses must be known by then.
* **Horizon (h)** - how many weeks ahead: 1, 2 or 4. The **target** is the observed price of week *t + h*.
* **Naive forecast** - the latest available price. Every other model must justify itself against it.
* **Skill** - `1 − MAE(model) / MAE(naive)`. Positive means better than naive; 0.10 means 10% lower error.
* **MAE** - mean absolute error in KES/kg. **WAPE** - total absolute error divided by total actual price (scale-free).
* **Scenario** - how messy the data are: *clean*, *realistic*, *stressed*. **Handling on/off** - whether the pipeline cleans the data (causally) before modelling.
* **E0 … E4** - cumulative feature sets: E0 price history + calendar → E4 adds supply/demand, weather, logistics and cross-market signals.
* **Leakage** - using, directly or indirectly, information that would not have been available at the forecast origin.
* **Regime shift** - a lasting change in how the market works (here: transport costs jump at week 140 and never come back).